In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
def pull_cc_with_exclusions(approvedate_start: str, approvedate_end: str):
    """
    Pull credit card approvals within [approvedate_start, approvedate_end)
    and apply business exclusion + inclusion filters.

    Parameters
    ----------
    approvedate_start : str  – inclusive lower bound, e.g. '2025-06-01'
    approvedate_end   : str  – exclusive upper bound, e.g. '2026-07-01'

    Returns
    -------
    df_cc_clean : pyspark DataFrame after all exclusions/inclusions
    """
    # ========================================================================
    # Step 1: Pull raw data
    # ========================================================================
    df_cc = spark.sql(f"""
        SELECT *
        FROM mis_automation.silver_credit_card.vw_credit_card_history
        WHERE APPROVEDATE >= '{approvedate_start}'
          AND APPROVEDATE <  '{approvedate_end}'
    """)

    # Force consistent types for SNAPSHOT_DATE, MONTH, YEAR across partitions
    df_cc = (df_cc
        .withColumn("SNAPSHOT_DATE", F.col("SNAPSHOT_DATE").cast("date"))
        .withColumn("MONTH", F.col("MONTH").cast("int"))
        .withColumn("YEAR", F.col("YEAR").cast("int"))
    )

    # Force ALL columns nullable to prevent ArrowInvalid during .toPandas()
    # Source view has inconsistent nullability for VIP across partition files;
    # unionByName with an empty all-nullable DF is optimizer-proof on Spark Connect
    from pyspark.sql.types import StructType, StructField
    _nullable = StructType([StructField(f.name, f.dataType, True) for f in df_cc.schema.fields])
    df_cc = df_cc.unionByName(spark.createDataFrame([], _nullable))

    raw_rows = df_cc.count()
    raw_cards = df_cc.select('CREDITCARDNO').distinct().count()
    print(f"Before exclusions — Rows: {raw_rows:,} | Unique cards: {raw_cards:,}")

    # Monthly approved card count (before exclusions)
    monthly_approved = (
        df_cc
        .withColumn("approve_month", F.date_format(F.col("APPROVEDATE"), "yyyy-MM"))
        .groupBy("approve_month")
        .agg(F.countDistinct("CREDITCARDNO").alias("approved_cards"))
        .orderBy("approve_month")
    )
    print("\nMonthly approved card count (before exclusions):")
    monthly_approved.show(truncate=False)

    # ====================================================================
    # Step 2: Compute per-card OPEN snapshot count (needed for Exclusion 9)
    # ====================================================================
    w = Window.partitionBy("CREDITCARDNO")
    df_cc = df_cc.withColumn(
        "open_count",
        F.sum(F.when(F.col("CURRENT_STATUS") == "1.OPEN", 1).otherwise(0)).over(w)
    )

    # ====================================================================
    # Step 3: Apply business exclusion filters
    # ====================================================================
    exclude_cond = (
        (F.col("VIP") == "1") |                                                              # Exclusion 3a: VIP customers
        (F.upper(F.trim(F.col("RULING_FAMILY_FLAG"))) == "Y") |                             # Exclusion 3b: Ruling family
        # (F.col("AECB_DATA").isNull()) |                                                  # Exclusion 4: No AECB data (disabled)
        (F.col("PRIMARYCARD") == 0) |                                                      # Exclusion 7: Supplementary cards
        # (F.col("open_count") < 3) |                                                        # Exclusion 9: Fewer than 3 OPEN snapshots
        (F.upper(F.trim(F.col("F_SEGMENT_NEW"))) == "SALARIED-IM") |                        # Exclusion 10a: Salaried-IM
        (F.upper(F.trim(F.col("SEGMENT_POLICY"))).isin(
            "PL CUSTOMERS", "AUTO-LOAN CUSTOMERS", "MORTGAGE CXMS", "STAFF"))               # Exclusion 10b: Unwanted segment policies
    )

    inclusive_cond = (
        (F.col("CLASSCODE").isNull() | ~F.trim(F.col("CLASSCODE")).isin("STAFF", "SME"))
        & (F.upper(F.trim(F.col("CARD_BUSINESS_SEGMENT"))).isin("CARDS"))
        & (F.upper(F.trim(F.col("CUSTOMER_SEGMENT"))) == "SALARIED")
    )

    df_cc_clean = df_cc.filter(~exclude_cond).filter(inclusive_cond)

    clean_rows = df_cc_clean.count()
    clean_cards = df_cc_clean.select('CREDITCARDNO').distinct().count()
    clean_refs = df_cc_clean.select('REFERENCE_NUMBER').distinct().count()
    print(f"After exclusions  — Rows: {clean_rows:,} | Unique cards: {clean_cards:,} | Unique refs: {clean_refs:,}")
    print(f"Excluded: {raw_rows - clean_rows:,} rows, {raw_cards - clean_cards:,} cards")

    # Monthly approved card count (after exclusions)
    monthly_approved_clean = (
        df_cc_clean
        .withColumn("approve_month", F.date_format(F.col("APPROVEDATE"), "yyyy-MM"))
        .groupBy("approve_month")
        .agg(F.countDistinct("CREDITCARDNO").alias("approved_cards"))
        .orderBy("approve_month")
    )
    print("\nMonthly new approved card count (after exclusions):")
    monthly_approved_clean.show(truncate=False)

    return df_cc_clean


# Example usage
df_cc_pd = pull_cc_with_exclusions('2025-06-01', '2026-07-01').toPandas()

In [ ]:
df_cc_pd['FTS_Ref_No'].value_counts()

In [ ]:
# Get unique reference numbers from the credit card pull
df_cc_pd['REFERENCE_NUMBER'] = pd.to_numeric(df_cc_pd['REFERENCE_NUMBER'], errors='coerce').fillna(-1).astype(int).astype(str)
ref_numbers = [str(int(r)) for r in df_cc_pd['REFERENCE_NUMBER'].unique().tolist()]
print(f"Unique reference numbers to look up: {len(ref_numbers):,}")

# Load AECB feature store and filter to matching reference numbers
aecb = spark.sql("""
    SELECT Reference_Number, file_type
    FROM mis_automation.silver_bureauone.slv_credit_card_aecb_feature_store
""")

aecb_filtered = (
    aecb
    .withColumn("Reference_Number", F.col("Reference_Number").cast("string"))
    .filter(F.col("Reference_Number").isin(ref_numbers))
    .filter(F.col("file_type") == "Thin File")
)

aecb_thin_refs = aecb_filtered.select("Reference_Number").distinct().toPandas()
thin_ref_set = set(aecb_thin_refs['Reference_Number'].astype(str))
print(f"Thin File reference numbers found in AECB: {len(thin_ref_set):,}")

# Coverage check
print(f"Coverage: {len(thin_ref_set):,} / {len(ref_numbers):,} reference numbers are Thin File ({len(thin_ref_set)/len(ref_numbers):.1%})")

# Filter df_cc_pd to only Thin File reference numbers
df_cc_pd['REFERENCE_NUMBER'] = df_cc_pd['REFERENCE_NUMBER'].astype(float).astype(int).astype(str)
df_thin = df_cc_pd[df_cc_pd['REFERENCE_NUMBER'].isin(thin_ref_set)].reset_index(drop=True)
print(f"\nThin File df_cc_pd shape: {df_thin.shape}")

In [ ]:
df_thin.FTS_Ref_No.nunique()#13012
df_thin.REFERENCE_NUMBER.nunique()#16611
df_thin.CRNNO.nunique()#16626

In [ ]:
# ── Bad flag per row: DELINQUENCYBUCKET > 3 → 90 DPD ──────────────────────────
df_thin['DELINQUENCYBUCKET'] = pd.to_numeric(df_thin['DELINQUENCYBUCKET'], errors='coerce')
df_thin['is_bad'] = (df_thin['DELINQUENCYBUCKET'] > 3).astype(int)
df_thin['snapshot_month'] = pd.to_datetime(df_thin['SNAPSHOT_DATE']).dt.to_period('M')

# ── Good flag: rolling 4-month window per card × snapshot month ───────────────
def compute_good_flag_per_row(df):
    """
    For every (CREDITCARDNO, snapshot_month) pair, look back 4 calendar months
    [M-3 … M].  If the card has data in all 4 months AND max delinquency
    bucket == 0 across those months → Good (is_good = 1) for that snapshot.

    A card can be Good in one snapshot month and Bad/Unknown in another.

    Returns
    -------
    good_lookup : DataFrame with columns [CREDITCARDNO, snapshot_month, is_good]
    """
    sub = df.copy()
    sub['snapshot_month'] = pd.to_datetime(sub['SNAPSHOT_DATE']).dt.to_period('M')
    sub['DELINQUENCYBUCKET'] = pd.to_numeric(sub['DELINQUENCYBUCKET'], errors='coerce')

    # One row per card × snapshot_month: max delinquency in that month
    card_snap = (
        sub.groupby(['CREDITCARDNO', 'snapshot_month'], as_index=False)
        .agg(max_bucket=('DELINQUENCYBUCKET', 'max'))
    )

    all_eval_months = sorted(card_snap['snapshot_month'].unique())
    print(f"  Evaluating Good flag across {len(all_eval_months)} months")
    print(f"  Unique card × month combos: {len(card_snap):,}")

    # For each card × eval snapshot, build 4 target calendar months [M-3…M]
    target_frames = []
    for offset in range(4):
        tmp = card_snap[['CREDITCARDNO', 'snapshot_month']].copy()
        tmp['eval_month'] = tmp['snapshot_month']
        tmp['target_month'] = tmp['snapshot_month'] - offset
        target_frames.append(tmp[['CREDITCARDNO', 'eval_month', 'target_month']])
    targets = pd.concat(target_frames, ignore_index=True)

    # Left-join to get actual bucket at each target month (from ALL months)
    targets = targets.merge(
        card_snap.rename(columns={'snapshot_month': 'target_month'}),
        on=['CREDITCARDNO', 'target_month'],
        how='left'
    )

    # Aggregate per card × eval_month
    eval_stats = (
        targets.groupby(['CREDITCARDNO', 'eval_month'])
        .agg(
            n_months_with_data=('max_bucket', 'count'),
            max_bucket_4m=('max_bucket', 'max')
        )
        .reset_index()
    )

    eval_stats['status'] = np.where(
        eval_stats['n_months_with_data'] < 4, 'Unknown',
        np.where(eval_stats['max_bucket_4m'] == 0, 'Good', 'Not Good')
    )

    eval_stats['is_good'] = (eval_stats['status'] == 'Good').astype(int)

    # Status distribution
    status_dist = eval_stats['status'].value_counts()
    print(f"  Status distribution (card × month level):")
    for s, c in status_dist.items():
        print(f"    {s}: {c:,}")

    good_lookup = eval_stats[['CREDITCARDNO', 'eval_month', 'is_good']].rename(
        columns={'eval_month': 'snapshot_month'}
    )
    return good_lookup


# ── Apply Good flag per card × snapshot month ─────────────────────────────────
print("Computing Good flag per card × snapshot month...\n")
good_lookup = compute_good_flag_per_row(df_thin)

# Merge is_good back to every row by (CREDITCARDNO, snapshot_month)
if 'is_good' in df_thin.columns:
    df_thin.drop(columns='is_good', inplace=True)

df_thin = df_thin.merge(
    good_lookup,
    on=['CREDITCARDNO', 'snapshot_month'],
    how='left'
)
df_thin['is_good'] = df_thin['is_good'].fillna(0).astype(int)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Total rows : {len(df_thin):,}")
print(f"is_bad  = 1: {df_thin['is_bad'].sum():,} rows")
print(f"is_good = 1: {df_thin['is_good'].sum():,} rows")
print(f"Neither    : {((df_thin['is_bad'] == 0) & (df_thin['is_good'] == 0)).sum():,} rows")
print(f"\nUnique cards — Bad : {df_thin.loc[df_thin['is_bad']==1, 'CREDITCARDNO'].nunique():,}")
print(f"Unique cards — Good: {df_thin.loc[df_thin['is_good']==1, 'CREDITCARDNO'].nunique():,}")
print(f"\nCards that are Good in some months and Bad in others: "
      f"{len(set(df_thin.loc[df_thin['is_bad']==1, 'CREDITCARDNO'].unique()) & set(df_thin.loc[df_thin['is_good']==1, 'CREDITCARDNO'].unique())):,}")

In [ ]:
# ── Build universe: APPROVEDATE 2025-06 to 2026-02, SNAPSHOT 2026-06/07, is_bad=1 or is_good=1 ──

df_thin['APPROVEDATE'] = pd.to_datetime(df_thin['APPROVEDATE'])
df_thin['SNAPSHOT_DATE'] = pd.to_datetime(df_thin['SNAPSHOT_DATE'])

universe = df_thin[
    # Approve date: June 2025 to Feb 2026 (inclusive)
    (df_thin['APPROVEDATE'] >= '2025-06-01') &
    (df_thin['APPROVEDATE'] < '2026-03-01') &
    # Snapshot date: June 2026 or July 2026
    (df_thin['SNAPSHOT_DATE'].dt.to_period('M').isin([pd.Period('2026-06', 'M'), pd.Period('2026-07', 'M')])) &
    # Only bad or good cases
    ((df_thin['is_bad'] == 1) | (df_thin['is_good'] == 1))
].copy()

print(f"Universe shape: {universe.shape}")
print(f"Unique cards:   {universe['CREDITCARDNO'].nunique():,}")
print(f"Unique refs:    {universe['REFERENCE_NUMBER'].nunique():,}")
print(f"\nApprove date range: {universe['APPROVEDATE'].min()} → {universe['APPROVEDATE'].max()}")
print(f"Snapshot months:    {sorted(universe['SNAPSHOT_DATE'].dt.to_period('M').unique())}")
print(f"\nis_bad=1 rows:  {(universe['is_bad']==1).sum():,}  | cards: {universe.loc[universe['is_bad']==1, 'CREDITCARDNO'].nunique():,}")
print(f"is_good=1 rows: {(universe['is_good']==1).sum():,}  | cards: {universe.loc[universe['is_good']==1, 'CREDITCARDNO'].nunique():,}")
print(f"Both bad & good cards: {len(set(universe.loc[universe['is_bad']==1, 'CREDITCARDNO'].unique()) & set(universe.loc[universe['is_good']==1, 'CREDITCARDNO'].unique())):,}")

In [ ]:
# Combine is_bad and is_good into a single 'target' column
universe['target'] = np.where(
    (universe['is_bad'] == 1) & (universe['is_good'] == 0), 1,
    np.where(
        (universe['is_bad'] == 0) & (universe['is_good'] == 1), 0,
        -1
    )
)

print(f"Target distribution:\n{universe['target'].value_counts().sort_index()}")
print(f"\nTotal rows: {len(universe):,}")
universe.head()

In [ ]:
# ── Pull FTS statement data and join to universe on FTS_Ref_No ─────────────────

# Get unique FTS_Ref_No values from universe for filtering
fts_refs = [str(r) for r in universe['FTS_Ref_No'].dropna().unique().tolist()]
print(f"Unique FTS_Ref_No values in universe: {len(fts_refs):,}")

# Create a temporary view of FTS_Ref_No values for push-down filtering
fts_refs_df = spark.createDataFrame([(r,) for r in fts_refs], ["FTS_Ref_No"])
fts_refs_df.createOrReplaceTempView("tmp_fts_refs")

# Pull FTS data with server-side filtering: snapshot month + FTS_Ref_No
df_fts_filtered = spark.sql("""
    SELECT f.*
    FROM mis_automation.silver_ftsconnect.vw_fts_stmt_data_message_details f
    INNER JOIN tmp_fts_refs r
        ON CAST(f.FTS_Ref_No AS STRING) = r.FTS_Ref_No
""")

print(f"FTS rows matching universe FTS_Ref_No: {df_fts_filtered.count():,}")
print(f"FTS unique FTS_Ref_No matched: {df_fts_filtered.select('FTS_Ref_No').distinct().count():,}")

# Convert to pandas
df_fts_pd = df_fts_filtered.toPandas()
df_fts_pd['FTS_Ref_No'] = df_fts_pd['FTS_Ref_No'].astype(str)

# ── Join FTS data to universe on FTS_Ref_No ───────────────────────────────────
universe['FTS_Ref_No'] = universe['FTS_Ref_No'].astype(str)

# universe_fts = universe.merge(df_fts_pd, on='FTS_Ref_No', how='left', suffixes=('', '_fts'))

# print(f"\nUniverse shape before join: {universe.shape}")
# print(f"Universe+FTS shape after join: {universe_fts.shape}")
# print(f"FTS columns added: {len(df_fts_pd.columns) - 1}")

# # Coverage check
# matched = universe_fts[universe_fts[[c for c in df_fts_pd.columns if c != 'FTS_Ref_No'][0]].notna()]['FTS_Ref_No'].nunique()
# total = universe['FTS_Ref_No'].dropna().nunique()
# print(f"\nFTS coverage: {matched:,} / {total:,} FTS_Ref_No matched ({matched/total:.1%})")

# # ── Quick Good vs Bad comparison on FTS numeric features ──────────────────────
# print(f"\n{'='*60}")
# print("Quick stats: Good vs Bad across FTS numeric columns")
# print(f"{'='*60}")

# fts_cols = [c for c in df_fts_pd.columns if c != 'FTS_Ref_No']
# numeric_fts_cols = universe_fts[fts_cols].select_dtypes(include='number').columns.tolist()

# if numeric_fts_cols:
#     comparison = (
#         universe_fts
#         .assign(label=np.where(universe_fts['is_bad'] == 1, 'Bad', 'Good'))
#         .groupby('label')[numeric_fts_cols]
#         .agg(['mean', 'median', 'std', 'count'])
#     )
#     # Flatten multi-level columns for readability
#     comparison.columns = ['_'.join(col).strip() for col in comparison.columns]
#     print(f"\nNumeric FTS features found: {len(numeric_fts_cols)}")
#     display(comparison.T)
# else:
#     print("No numeric FTS columns found. Showing value counts for categorical columns:")
#     cat_fts_cols = [c for c in fts_cols if c in universe_fts.columns][:10]  # first 10
#     for col in cat_fts_cols:
#         print(f"\n--- {col} ---")
#         print(universe_fts.groupby(np.where(universe_fts['is_bad'] == 1, 'Bad', 'Good'))[col].value_counts().unstack(fill_value=0))

# print(f"\nFTS columns available: {fts_cols}")

In [ ]:
# ── Aggregate FTS statement data per FTS_Ref_No × month ───────────────────────

# ── Pre-aggregation missing-value audit ────────────────────────────────────────
# Check the columns that will be grouped-by or aggregated for NaN / missing values
_audit_cols = ['FTS_Ref_No', 'SLD_Value_Date', 'SLD_Transaction_Amount', 'SLD_Debit_Credit_Mask']
print("=" * 70)
print("PRE-AGGREGATION MISSING VALUE AUDIT")
print("=" * 70)
for col in _audit_cols:
    n_null = df_fts_pd[col].isna().sum()
    n_empty = (df_fts_pd[col].astype(str).str.strip() == '').sum() if df_fts_pd[col].dtype == object else 0
    pct = (n_null + n_empty) / len(df_fts_pd) * 100
    print(f"  {col:35s} | NaN: {n_null:>7,} | Empty str: {n_empty:>7,} | Total missing: {n_null + n_empty:>7,} ({pct:.2f}%)")
print()

# ── Handle missing values BEFORE aggregation ──────────────────────────────────
# 1) SLD_Transaction_Amount: coerce to numeric; fill NaN with 0
#    Rationale: a missing amount is not "unknown" — it means no monetary value;
#    filling with 0 is safe for sum/count aggregations.
df_fts_pd['SLD_Transaction_Amount'] = pd.to_numeric(df_fts_pd['SLD_Transaction_Amount'], errors='coerce')
_n_amt_na = df_fts_pd['SLD_Transaction_Amount'].isna().sum()
print(f"SLD_Transaction_Amount NaN after coerce (before fill): {_n_amt_na:,}")
df_fts_pd['SLD_Transaction_Amount'] = df_fts_pd['SLD_Transaction_Amount'].fillna(0)

# 2) SLD_Value_Date → month: drop rows where date is unparseable
#    Rationale: we cannot assign a transaction to a month if the date is missing;
#    dropping is safer than inventing a month (which would pollute aggregations).
df_fts_pd['month'] = pd.to_datetime(df_fts_pd['SLD_Value_Date'], errors='coerce').dt.to_period('M')
_n_month_na = df_fts_pd['month'].isna().sum()
print(f"Rows with unparseable SLD_Value_Date (month is NaT): {_n_month_na:,}")
if _n_month_na > 0:
    print(f"  → Dropping {_n_month_na:,} rows where month could not be derived.")
    df_fts_pd = df_fts_pd.dropna(subset=['month']).reset_index(drop=True)

# 3) SLD_Debit_Credit_Mask: rows that are neither 'C' nor 'D' will naturally be
#    excluded by the credit_mask / debit_mask filters below — audit them here.
_dc_values = df_fts_pd['SLD_Debit_Credit_Mask'].value_counts(dropna=False)
print(f"\nSLD_Debit_Credit_Mask value distribution:")
print(_dc_values.to_string())
_n_neither = (~df_fts_pd['SLD_Debit_Credit_Mask'].isin(['C', 'D'])).sum()
if _n_neither > 0:
    print(f"  ⚠ {_n_neither:,} rows are neither C nor D — they will be excluded from inflow/outflow.")

# 4) FTS_Ref_No: drop rows with missing group-by key
_n_ref_na = df_fts_pd['FTS_Ref_No'].isna().sum()
print(f"\nFTS_Ref_No missing: {_n_ref_na:,}")
if _n_ref_na > 0:
    print(f"  → Dropping {_n_ref_na:,} rows with missing FTS_Ref_No (cannot group).")
    df_fts_pd = df_fts_pd.dropna(subset=['FTS_Ref_No']).reset_index(drop=True)

print(f"\nClean df_fts_pd shape for aggregation: {df_fts_pd.shape}")
print("=" * 70)

# Credit (C) = inflow, Debit (D) = outflow
credit_mask = df_fts_pd['SLD_Debit_Credit_Mask'] == 'C'
debit_mask  = df_fts_pd['SLD_Debit_Credit_Mask'] == 'D'

inflow = (
    df_fts_pd[credit_mask]
    .groupby(['FTS_Ref_No', 'month'], as_index=False)
    .agg(inflow_amount=('SLD_Transaction_Amount', 'sum'),
         inflow_txn_count=('SLD_Transaction_Amount', 'count'))
)

outflow = (
    df_fts_pd[debit_mask]
    .groupby(['FTS_Ref_No', 'month'], as_index=False)
    .agg(outflow_amount=('SLD_Transaction_Amount', 'sum'),
         outflow_txn_count=('SLD_Transaction_Amount', 'count'))
)

# Merge inflow and outflow
FTS_Ref_No_monthly = inflow.merge(outflow, on=['FTS_Ref_No', 'month'], how='outer').fillna(0)
FTS_Ref_No_monthly['net_flow'] = FTS_Ref_No_monthly['inflow_amount'] - FTS_Ref_No_monthly['outflow_amount']
FTS_Ref_No_monthly = FTS_Ref_No_monthly.sort_values(['FTS_Ref_No', 'month']).reset_index(drop=True)

# ── Aggregate monthly features to per-FTS_Ref_No level ────────────────────────
universe_fts = (
    FTS_Ref_No_monthly
    .groupby('FTS_Ref_No', as_index=False)
    .agg(
        total_inflow=('inflow_amount', 'sum'),
        total_outflow=('outflow_amount', 'sum'),
        total_net_flow=('net_flow', 'sum'),
        avg_monthly_inflow=('inflow_amount', 'mean'),
        avg_monthly_outflow=('outflow_amount', 'mean'),
        avg_monthly_net_flow=('net_flow', 'mean'),
        std_monthly_inflow=('inflow_amount', 'std'),
        std_monthly_outflow=('outflow_amount', 'std'),
        std_monthly_net_flow=('net_flow', 'std'),
        max_monthly_inflow=('inflow_amount', 'max'),
        max_monthly_outflow=('outflow_amount', 'max'),
        min_monthly_inflow=('inflow_amount', 'min'),
        min_monthly_outflow=('outflow_amount', 'min'),
        total_inflow_txn_count=('inflow_txn_count', 'sum'),
        total_outflow_txn_count=('outflow_txn_count', 'sum'),
        avg_monthly_inflow_txns=('inflow_txn_count', 'mean'),
        avg_monthly_outflow_txns=('outflow_txn_count', 'mean'),
        n_active_months=('month', 'nunique'),
    )
)

# Fill NaN std values (single-month accounts) with 0
universe_fts[['std_monthly_inflow', 'std_monthly_outflow', 'std_monthly_net_flow']] = (
    universe_fts[['std_monthly_inflow', 'std_monthly_outflow', 'std_monthly_net_flow']].fillna(0)
)

# Derived ratios
universe_fts['outflow_inflow_ratio'] = np.where(
    universe_fts['total_inflow'] > 0,
    universe_fts['total_outflow'] / universe_fts['total_inflow'],
    0
)

print(f"FTS_Ref_No_monthly shape: {FTS_Ref_No_monthly.shape}")
print(f"universe_fts (per FTS_Ref_No) shape: {universe_fts.shape}")
print(f"\nFeatures generated: {list(universe_fts.columns)}")
universe_fts.head()

In [ ]:
# Merge FTS aggregated features into universe on FTS_Ref_No
universe = universe.merge(universe_fts, on='FTS_Ref_No', how='left')

print(f"Universe shape after merging FTS features: {universe.shape}")
print(f"FTS feature coverage: {universe['total_inflow'].notna().sum():,} / {len(universe):,} rows ({universe['total_inflow'].notna().mean():.1%})")
universe.head()

In [ ]:
'target'  in universe.columns

In [ ]:
# ── Correlation analysis: FTS features vs target ─────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

features = [
    'total_inflow', 'total_outflow', 'total_net_flow',
    'avg_monthly_inflow', 'avg_monthly_outflow', 'avg_monthly_net_flow',
    'std_monthly_inflow', 'std_monthly_outflow', 'std_monthly_net_flow',
    'max_monthly_inflow', 'max_monthly_outflow',
    'min_monthly_inflow', 'min_monthly_outflow',
    'total_inflow_txn_count', 'total_outflow_txn_count',
    'avg_monthly_inflow_txns', 'avg_monthly_outflow_txns',
    'n_active_months', 'outflow_inflow_ratio'
]

# Filter to rows with FTS data and valid target (0 or 1)
df_corr = universe[universe['total_inflow'].notna() & universe['target'].isin([0, 1])].copy()
print(f"Rows for correlation analysis: {len(df_corr):,}")
print(f"  target=1 (Bad):  {(df_corr['target']==1).sum():,}")
print(f"  target=0 (Good): {(df_corr['target']==0).sum():,}")

# ── 1. Point-biserial correlation (Pearson between binary & continuous) ───────
corr_results = []
for col in features:
    valid = df_corr[[col, 'target']].dropna()
    if len(valid) < 10:
        continue
    r, p = stats.pointbiserialr(valid['target'], valid[col])
    # Also compute mean for Good vs Bad
    mean_good = valid.loc[valid['target'] == 0, col].mean()
    mean_bad  = valid.loc[valid['target'] == 1, col].mean()
    corr_results.append({
        'feature': col,
        'correlation': r,
        'abs_corr': abs(r),
        'p_value': p,
        'significant': '✅' if p < 0.05 else '❌',
        'mean_good': mean_good,
        'mean_bad': mean_bad,
        'pct_diff': ((mean_bad - mean_good) / mean_good * 100) if mean_good != 0 else np.nan
    })

corr_df = pd.DataFrame(corr_results).sort_values('abs_corr', ascending=False).reset_index(drop=True)

print(f"\n{'='*90}")
print(f"POINT-BISERIAL CORRELATION: FTS features vs target (1=Bad, 0=Good)")
print(f"{'='*90}")
print(f"{'Feature':40s} {'Corr':>8s} {'p-value':>12s} {'Sig':>4s} {'Mean Good':>12s} {'Mean Bad':>12s} {'%Diff':>8s}")
print(f"{'-'*90}")
for _, row in corr_df.iterrows():
    strength = '🔴' if row['abs_corr'] >= 0.15 else ('🟡' if row['abs_corr'] >= 0.05 else '🟢')
    print(f"{strength} {row['feature']:38s} {row['correlation']:>8.4f} {row['p_value']:>12.2e} {row['significant']:>4s} "
          f"{row['mean_good']:>12.1f} {row['mean_bad']:>12.1f} {row['pct_diff']:>7.1f}%")

print(f"\n🔴 |r| ≥ 0.15  🟡 0.05 ≤ |r| < 0.15  🟢 |r| < 0.05")

# ── 2. Correlation heatmap ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# 2a. Feature-to-target correlations (bar chart)
ax1 = axes[0]
corr_df_sorted = corr_df.sort_values('correlation')
colors = ['#d32f2f' if c > 0 else '#1976d2' for c in corr_df_sorted['correlation']]
ax1.barh(corr_df_sorted['feature'], corr_df_sorted['correlation'], color=colors, edgecolor='white')
ax1.set_xlabel('Point-Biserial Correlation with Target', fontsize=12)
ax1.set_title('Feature Correlation with Target\n(positive = higher value → more likely Bad)', fontsize=13)
ax1.axvline(x=0, color='black', linewidth=0.8)
ax1.axvline(x=0.05, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
ax1.axvline(x=-0.05, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
plt.tight_layout()

# 2b. Feature-to-feature + target correlation heatmap
ax2 = axes[1]
corr_matrix = df_corr[features + ['target']].corr()
sns.heatmap(corr_matrix[['target']].sort_values('target', ascending=False),
            annot=True, fmt='.3f', cmap='RdBu_r', center=0, vmin=-0.3, vmax=0.3,
            ax=ax2, cbar_kws={'label': 'Correlation'})
ax2.set_title('Correlation with Target', fontsize=13)
plt.tight_layout()
plt.show()

# ── 3. Box plots for top features ────────────────────────────────────────────
top_features = corr_df.head(6)['feature'].tolist()
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, col in enumerate(top_features):
    ax = axes[idx // 3][idx % 3]
    data_good = df_corr.loc[df_corr['target'] == 0, col].dropna()
    data_bad  = df_corr.loc[df_corr['target'] == 1, col].dropna()
    ax.boxplot([data_good, data_bad], labels=['Good (0)', 'Bad (1)'],
               patch_artist=True,
               boxprops=dict(facecolor='lightblue'),
               showfliers=False)  # hide outliers for cleaner view
    r_val = corr_df.loc[corr_df['feature'] == col, 'correlation'].values[0]
    ax.set_title(f'{col}\n(r = {r_val:.4f})', fontsize=11)
    ax.set_ylabel('Value')
fig.suptitle('Top 6 Correlated FTS Features: Good vs Bad Distribution', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# ── 4. Summary table ─────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
n_sig = (corr_df['p_value'] < 0.05).sum()
n_strong = (corr_df['abs_corr'] >= 0.15).sum()
n_moderate = ((corr_df['abs_corr'] >= 0.05) & (corr_df['abs_corr'] < 0.15)).sum()
print(f"  Statistically significant (p < 0.05): {n_sig} / {len(corr_df)}")
print(f"  Strong correlation (|r| ≥ 0.15):      {n_strong}")
print(f"  Moderate correlation (0.05 ≤ |r| < 0.15): {n_moderate}")
print(f"  Weak correlation (|r| < 0.05):         {len(corr_df) - n_strong - n_moderate}")

display(corr_df)

In [ ]:
# ── Random Forest: FTS features → target ──────────────────────────────────────
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, f1_score, precision_score, recall_score
)
import matplotlib.pyplot as plt
import seaborn as sns

# ── Feature list & target ─────────────────────────────────────────────────────
features = [
    'total_inflow', 'total_outflow', 'total_net_flow',
    'avg_monthly_inflow', 'avg_monthly_outflow', 'avg_monthly_net_flow',
    'std_monthly_inflow', 'std_monthly_outflow', 'std_monthly_net_flow',
    'max_monthly_inflow', 'max_monthly_outflow',
    'min_monthly_inflow', 'min_monthly_outflow',
    'total_inflow_txn_count', 'total_outflow_txn_count',
    'avg_monthly_inflow_txns', 'avg_monthly_outflow_txns',
    'n_active_months', 'outflow_inflow_ratio'
]

# Filter to rows with FTS data and valid target (0 or 1)
df_model = universe[universe['total_inflow'].notna() & universe['target'].isin([0, 1])].copy()
X = df_model[features].fillna(0)
y = df_model['target']

print(f"Modelling dataset: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Target distribution:\n{y.value_counts().sort_index()}")
print(f"Bad rate: {y.mean():.4f}")

# ── Deduplicate to ONE row per card (latest snapshot) ─────────────────────────
# The universe has ~1.94 rows/card (dual snapshots 2026-06 & 2026-07).
# FTS features are identical per card, so duplicates inflate metrics.
df_model = df_model.sort_values('SNAPSHOT_DATE', ascending=False)
df_model = df_model.drop_duplicates(subset='CREDITCARDNO', keep='first').reset_index(drop=True)
X = df_model[features].fillna(0)
y = df_model['target']
print(f"After dedup to 1 row/card: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Target distribution (deduped):\n{y.value_counts().sort_index()}")
print(f"Bad rate: {y.mean():.4f}")

# ── Card-level Train / Test split (prevents same card in both sets) ───────────
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df_model['CREDITCARDNO']))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print(f"\nTrain: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
print(f"Train bad rate: {y_train.mean():.4f}  |  Test bad rate: {y_test.mean():.4f}")
print(f"Card overlap check: {len(set(df_model.loc[train_idx, 'CREDITCARDNO']) & set(df_model.loc[test_idx, 'CREDITCARDNO']))} cards in both (should be 0)")

# ── Train Random Forest ───────────────────────────────────────────────────────
mlflow.sklearn.autolog(log_models=True, silent=True)

with mlflow.start_run(run_name="RF_FTS_features") as run:
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)

    # ── Predictions ───────────────────────────────────────────────────────────
    y_train_pred = rf.predict(X_train)
    y_test_pred  = rf.predict(X_test)
    y_train_prob = rf.predict_proba(X_train)[:, 1]
    y_test_prob  = rf.predict_proba(X_test)[:, 1]

    # ── Metrics ───────────────────────────────────────────────────────────────
    metrics = {}
    for split, y_true, y_pred, y_prob in [
        ('train', y_train, y_train_pred, y_train_prob),
        ('test',  y_test,  y_test_pred,  y_test_prob)
    ]:
        metrics[f'{split}_accuracy']  = accuracy_score(y_true, y_pred)
        metrics[f'{split}_precision'] = precision_score(y_true, y_pred, zero_division=0)
        metrics[f'{split}_recall']    = recall_score(y_true, y_pred, zero_division=0)
        metrics[f'{split}_f1']        = f1_score(y_true, y_pred, zero_division=0)
        metrics[f'{split}_auc']       = roc_auc_score(y_true, y_prob)

    mlflow.log_metrics(metrics)

    # ── Print performance ─────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("RANDOM FOREST PERFORMANCE")
    print("=" * 70)
    for split, y_true, y_pred in [('TRAIN', y_train, y_train_pred),
                                   ('TEST',  y_test,  y_test_pred)]:
        print(f"\n{'─'*30} {split} {'─'*30}")
        print(f"  Accuracy:  {metrics[f'{split.lower()}_accuracy']:.4f}")
        print(f"  Precision: {metrics[f'{split.lower()}_precision']:.4f}")
        print(f"  Recall:    {metrics[f'{split.lower()}_recall']:.4f}")
        print(f"  F1 Score:  {metrics[f'{split.lower()}_f1']:.4f}")
        print(f"  AUC-ROC:   {metrics[f'{split.lower()}_auc']:.4f}")
        print(f"\n  Classification Report:")
        print(classification_report(y_true, y_pred, target_names=['Good (0)', 'Bad (1)']))

    # ── Feature Importance ────────────────────────────────────────────────────
    feat_imp = pd.DataFrame({
        'feature': features,
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=False).reset_index(drop=True)

    print("\n" + "=" * 70)
    print("FEATURE IMPORTANCE (Gini / MDI)")
    print("=" * 70)
    for _, row in feat_imp.iterrows():
        bar = '█' * int(row['importance'] * 100)
        print(f"  {row['feature']:35s} {row['importance']:.4f}  {bar}")

    # ── Plots ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))

    # 1. Feature importance bar chart
    ax = axes[0]
    ax.barh(feat_imp['feature'][::-1], feat_imp['importance'][::-1], color='steelblue')
    ax.set_xlabel('Importance (Gini)')
    ax.set_title('Feature Importance')

    # 2. Confusion matrix – Test
    ax = axes[1]
    cm = confusion_matrix(y_test, y_test_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Good', 'Bad'], yticklabels=['Good', 'Bad'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'Test Confusion Matrix (AUC={metrics["test_auc"]:.3f})')

    # 3. ROC Curve – Train & Test
    from sklearn.metrics import roc_curve
    ax = axes[2]
    for label, y_true, y_prob, ls in [
        ('Train', y_train, y_train_prob, '--'),
        ('Test',  y_test,  y_test_prob,  '-')
    ]:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_val = roc_auc_score(y_true, y_prob)
        ax.plot(fpr, tpr, ls, label=f'{label} (AUC={auc_val:.3f})')
    ax.plot([0, 1], [0, 1], 'k:', alpha=0.4)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('ROC Curve'); ax.legend()

    plt.tight_layout()
    mlflow.log_figure(fig, 'rf_performance.png')
    plt.show()

    print(f"\n✅ MLflow run logged: {run.info.run_id}")

display(feat_imp)

In [ ]:
universe.groupby(['FTS_Ref_No'])['CREDITCARDNO'].count().value_counts()

In [ ]:
# ============================================================================
# Per FTS_Ref_No: SAL stability analysis
#   1. Monthly SAL amount variance (how stable is the salary amount?)
#   2. Monthly SAL posting-day variance (how stable is the pay day?)
# ============================================================================

# Filter to SAL transactions only
sal_df = df_fts_pd[df_fts_pd['SLD_TRAN_TYPE'] == 'SAL'].copy()

# Parse dates and amounts
sal_df['SLD_Posting_Date'] = pd.to_datetime(sal_df['SLD_Posting_Date'], errors='coerce')
sal_df['SLD_Transaction_Amount'] = pd.to_numeric(sal_df['SLD_Transaction_Amount'], errors='coerce')
sal_df['posting_month'] = sal_df['SLD_Posting_Date'].dt.to_period('M')
sal_df['posting_day'] = sal_df['SLD_Posting_Date'].dt.day

# --- Per FTS_Ref_No per month: sum SAL amount & average posting day -----------------
FTS_Ref_No_month_sal = (
    sal_df
    .groupby(['FTS_Ref_No', 'posting_month'], as_index=False)
    .agg(
        monthly_sal_amount=('SLD_Transaction_Amount', 'sum'),
        avg_posting_day=('posting_day', 'mean'),
        sal_txn_count=('SLD_Transaction_Amount', 'count')
    )
)

# --- Per FTS_Ref_No: variance / std / cv across months ------------------------------
sal_stability = (
    FTS_Ref_No_month_sal
    .groupby('FTS_Ref_No', as_index=False)
    .agg(
        num_sal_months=('posting_month', 'nunique'),
        mean_sal_amount=('monthly_sal_amount', 'mean'),
        std_sal_amount=('monthly_sal_amount', 'std'),
        var_sal_amount=('monthly_sal_amount', 'var'),
        min_sal_amount=('monthly_sal_amount', 'min'),
        max_sal_amount=('monthly_sal_amount', 'max'),
        mean_posting_day=('avg_posting_day', 'mean'),
        std_posting_day=('avg_posting_day', 'std'),
        var_posting_day=('avg_posting_day', 'var'),
        min_posting_day=('avg_posting_day', 'min'),
        max_posting_day=('avg_posting_day', 'max'),
    )
)

# Coefficient of Variation (CV) — lower = more stable
sal_stability['cv_sal_amount'] = (
    sal_stability['std_sal_amount'] / sal_stability['mean_sal_amount']
).replace([np.inf, -np.inf], np.nan)

sal_stability['cv_posting_day'] = (
    sal_stability['std_posting_day'] / sal_stability['mean_posting_day']
).replace([np.inf, -np.inf], np.nan)

# Day range (max - min posting day across months)
sal_stability['posting_day_range'] = (
    sal_stability['max_posting_day'] - sal_stability['min_posting_day']
)

# Fill NaN std/var for single-month accounts
for col in ['std_sal_amount', 'var_sal_amount', 'std_posting_day', 'var_posting_day',
            'cv_sal_amount', 'cv_posting_day', 'posting_day_range']:
    sal_stability[col] = sal_stability[col].fillna(0)

# Merge SAL stability features into universe_fts
universe_sal = universe.merge(sal_stability, on='FTS_Ref_No', how='left')

print(f"sal_stability shape: {sal_stability.shape}")
print(f"universe_sal shape after merge: {universe_sal.shape}")

# SAL data coverage report
n_universe = len(universe_sal)
n_with_sal = universe_sal['num_sal_months'].notna().sum()
pct_with_sal = n_with_sal / n_universe * 100
n_without_sal = n_universe - n_with_sal
pct_without_sal = n_without_sal / n_universe * 100

print(f"\n{'='*60}")
print(f"SAL DATA COVERAGE REPORT")
print(f"{'='*60}")
print(f"  Total universe rows:          {n_universe:,}")
print(f"  With valid SAL data:           {n_with_sal:,}  ({pct_with_sal:.2f}%)")
print(f"  Without SAL data (missing):    {n_without_sal:,}  ({pct_without_sal:.2f}%)")

# --- Per FTS_Ref_No: % of universe rows that have SAL information -------------
fts_sal_coverage = (
    universe_sal
    .groupby('FTS_Ref_No', as_index=False)
    .agg(
        total_rows=('num_sal_months', 'size'),
        rows_with_sal=('num_sal_months', lambda x: x.notna().sum())
    )
)
fts_sal_coverage['pct_with_sal'] = (
    fts_sal_coverage['rows_with_sal'] / fts_sal_coverage['total_rows'] * 100
).round(2)
fts_sal_coverage = fts_sal_coverage.sort_values('pct_with_sal', ascending=False).reset_index(drop=True)

print(f"\n{'='*70}")
print(f"SAL COVERAGE PER FTS_Ref_No")
print(f"{'='*70}")
print(f"  Total unique FTS_Ref_No:       {len(fts_sal_coverage):,}")
print(f"  With SAL data (pct > 0):       {(fts_sal_coverage['pct_with_sal'] > 0).sum():,}")
print(f"  Without any SAL data (pct=0):  {(fts_sal_coverage['pct_with_sal'] == 0).sum():,}")
print(f"\n  Mean SAL coverage per FTS_Ref_No: {fts_sal_coverage['pct_with_sal'].mean():.2f}%")
print(f"  Median SAL coverage per FTS_Ref_No: {fts_sal_coverage['pct_with_sal'].median():.2f}%")
display(fts_sal_coverage)

print(f"\nSAL stability features: {list(sal_stability.columns)}")
sal_stability.head()

In [ ]:
# Merge FTS aggregated features into universe on FTS_Ref_No
universe = universe.merge(universe_fts, on='FTS_Ref_No', how='left')

print(f"Universe shape after merging FTS features: {universe.shape}")
universe.head()

In [ ]:
# ── Random Forest: SAL stability features → target ───────────────────────────
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, f1_score, precision_score, recall_score, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

# ── Feature list & target ─────────────────────────────────────────────────────
sal_features = [
    'num_sal_months', 'mean_sal_amount', 'std_sal_amount', 'var_sal_amount',
    'min_sal_amount', 'max_sal_amount', 'mean_posting_day', 'std_posting_day',
    'var_posting_day', 'min_posting_day', 'max_posting_day',
    'cv_sal_amount', 'cv_posting_day', 'posting_day_range'
]

# Filter to rows with SAL data and valid target (0 or 1)
df_model_sal = universe_sal[universe_sal['num_sal_months'].notna() & universe_sal['target'].isin([0, 1])].copy()
X = df_model_sal[sal_features].fillna(0)
y = df_model_sal['target']

print(f"Modelling dataset: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Target distribution:\n{y.value_counts().sort_index()}")
print(f"Bad rate: {y.mean():.4f}")

# ── Deduplicate to ONE row per card (latest snapshot) ─────────────────────────
df_model_sal = df_model_sal.sort_values('SNAPSHOT_DATE', ascending=False)
df_model_sal = df_model_sal.drop_duplicates(subset='CREDITCARDNO', keep='first').reset_index(drop=True)
X = df_model_sal[sal_features].fillna(0)
y = df_model_sal['target']
print(f"After dedup to 1 row/card: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Target distribution (deduped):\n{y.value_counts().sort_index()}")
print(f"Bad rate: {y.mean():.4f}")

# ── Card-level Train / Test split (prevents same card in both sets) ───────────
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df_model_sal['CREDITCARDNO']))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print(f"\nTrain: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
print(f"Train bad rate: {y_train.mean():.4f}  |  Test bad rate: {y_test.mean():.4f}")
print(f"Card overlap check: {len(set(df_model_sal.loc[train_idx, 'CREDITCARDNO']) & set(df_model_sal.loc[test_idx, 'CREDITCARDNO']))} cards in both (should be 0)")

# ── Train Random Forest ───────────────────────────────────────────────────────
mlflow.sklearn.autolog(log_models=True, silent=True)

with mlflow.start_run(run_name="RF_SAL_stability_features") as run:
    rf_sal = RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    rf_sal.fit(X_train, y_train)

    # ── Predictions ───────────────────────────────────────────────────────────
    y_train_pred = rf_sal.predict(X_train)
    y_test_pred  = rf_sal.predict(X_test)
    y_train_prob = rf_sal.predict_proba(X_train)[:, 1]
    y_test_prob  = rf_sal.predict_proba(X_test)[:, 1]

    # ── Metrics ───────────────────────────────────────────────────────────────
    metrics = {}
    for split, y_true, y_pred, y_prob in [
        ('train', y_train, y_train_pred, y_train_prob),
        ('test',  y_test,  y_test_pred,  y_test_prob)
    ]:
        metrics[f'{split}_accuracy']  = accuracy_score(y_true, y_pred)
        metrics[f'{split}_precision'] = precision_score(y_true, y_pred, zero_division=0)
        metrics[f'{split}_recall']    = recall_score(y_true, y_pred, zero_division=0)
        metrics[f'{split}_f1']        = f1_score(y_true, y_pred, zero_division=0)
        metrics[f'{split}_auc']       = roc_auc_score(y_true, y_prob)

    mlflow.log_metrics(metrics)

    # ── Print performance ─────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("RANDOM FOREST PERFORMANCE (SAL STABILITY FEATURES)")
    print("=" * 70)
    for split, y_true, y_pred in [('TRAIN', y_train, y_train_pred),
                                   ('TEST',  y_test,  y_test_pred)]:
        print(f"\n{'─'*30} {split} {'─'*30}")
        print(f"  Accuracy:  {metrics[f'{split.lower()}_accuracy']:.4f}")
        print(f"  Precision: {metrics[f'{split.lower()}_precision']:.4f}")
        print(f"  Recall:    {metrics[f'{split.lower()}_recall']:.4f}")
        print(f"  F1 Score:  {metrics[f'{split.lower()}_f1']:.4f}")
        print(f"  AUC-ROC:   {metrics[f'{split.lower()}_auc']:.4f}")
        print(f"\n  Classification Report:")
        print(classification_report(y_true, y_pred, target_names=['Good (0)', 'Bad (1)']))

    # ── Feature Importance ────────────────────────────────────────────────────
    feat_imp_sal = pd.DataFrame({
        'feature': sal_features,
        'importance': rf_sal.feature_importances_
    }).sort_values('importance', ascending=False).reset_index(drop=True)

    print("\n" + "=" * 70)
    print("FEATURE IMPORTANCE (Gini / MDI)")
    print("=" * 70)
    for _, row in feat_imp_sal.iterrows():
        bar = '█' * int(row['importance'] * 100)
        print(f"  {row['feature']:35s} {row['importance']:.4f}  {bar}")

    # ── Plots ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))

    # 1. Feature importance bar chart
    ax = axes[0]
    ax.barh(feat_imp_sal['feature'][::-1], feat_imp_sal['importance'][::-1], color='steelblue')
    ax.set_xlabel('Importance (Gini)')
    ax.set_title('SAL Feature Importance')

    # 2. Confusion matrix – Test
    ax = axes[1]
    cm = confusion_matrix(y_test, y_test_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Good', 'Bad'], yticklabels=['Good', 'Bad'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'Test Confusion Matrix (AUC={metrics["test_auc"]:.3f})')

    # 3. ROC Curve – Train & Test
    ax = axes[2]
    for label, y_true, y_prob, ls in [
        ('Train', y_train, y_train_prob, '--'),
        ('Test',  y_test,  y_test_prob,  '-')
    ]:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_val = roc_auc_score(y_true, y_prob)
        ax.plot(fpr, tpr, ls, label=f'{label} (AUC={auc_val:.3f})')
    ax.plot([0, 1], [0, 1], 'k:', alpha=0.4)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('ROC Curve'); ax.legend()

    plt.tight_layout()
    mlflow.log_figure(fig, 'rf_sal_performance.png')
    plt.show()

    print(f"\n✅ MLflow run logged: {run.info.run_id}")

display(feat_imp_sal)

In [ ]:
# ── Random Forest: Combined FTS + SAL features → target ──────────────────────
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, f1_score, precision_score, recall_score, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── Feature lists ─────────────────────────────────────────────────────────────
fts_features = [
    'total_inflow', 'total_outflow', 'total_net_flow',
    'avg_monthly_inflow', 'avg_monthly_outflow', 'avg_monthly_net_flow',
    'std_monthly_inflow', 'std_monthly_outflow', 'std_monthly_net_flow',
    'max_monthly_inflow', 'max_monthly_outflow',
    'min_monthly_inflow', 'min_monthly_outflow',
    'total_inflow_txn_count', 'total_outflow_txn_count',
    'avg_monthly_inflow_txns', 'avg_monthly_outflow_txns',
    'n_active_months', 'outflow_inflow_ratio'
]

sal_features = [
    'num_sal_months', 'mean_sal_amount', 'std_sal_amount', 'var_sal_amount',
    'min_sal_amount', 'max_sal_amount', 'mean_posting_day', 'std_posting_day',
    'var_posting_day', 'min_posting_day', 'max_posting_day',
    'cv_sal_amount', 'cv_posting_day', 'posting_day_range'
]

all_features = fts_features + sal_features
print(f"Total features: {len(all_features)}  (FTS: {len(fts_features)} + SAL: {len(sal_features)})")

# ── Filter to rows with FTS data and valid target ─────────────────────────────
df_model_all = universe_sal[
    universe_sal['total_inflow'].notna() & universe_sal['target'].isin([0, 1])
].copy()
print(f"Rows with FTS data & valid target: {len(df_model_all):,}")
print(f"  of which have SAL data: {df_model_all['num_sal_months'].notna().sum():,}")

# ── Deduplicate to ONE row per card (latest snapshot) ─────────────────────────
df_model_all = df_model_all.sort_values('SNAPSHOT_DATE', ascending=False)
df_model_all = df_model_all.drop_duplicates(subset='CREDITCARDNO', keep='first').reset_index(drop=True)

X = df_model_all[all_features].fillna(0)
y = df_model_all['target']
print(f"After dedup to 1 row/card: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Target distribution:\n{y.value_counts().sort_index()}")
print(f"Bad rate: {y.mean():.4f}")

# ── Card-level Train / Test split ─────────────────────────────────────────────
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df_model_all['CREDITCARDNO']))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print(f"\nTrain: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
print(f"Train bad rate: {y_train.mean():.4f}  |  Test bad rate: {y_test.mean():.4f}")
print(f"Card overlap: {len(set(df_model_all.loc[train_idx, 'CREDITCARDNO']) & set(df_model_all.loc[test_idx, 'CREDITCARDNO']))} (should be 0)")

# ── Train Random Forest (conservative to avoid overfitting) ───────────────────
mlflow.sklearn.autolog(log_models=True, silent=True)

with mlflow.start_run(run_name="RF_FTS_plus_SAL_combined") as run:
    rf_combined = RandomForestClassifier(
        n_estimators=300,
        max_depth=8,            # shallower than cell 14 (was 10) to reduce overfitting
        min_samples_split=20,   # stricter than cell 14 (was 10)
        min_samples_leaf=10,    # stricter than cell 14 (was 5)
        max_features='sqrt',    # explicit sqrt to limit each tree's view
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    rf_combined.fit(X_train, y_train)

    # ── Predictions ───────────────────────────────────────────────────────────
    y_train_pred = rf_combined.predict(X_train)
    y_test_pred  = rf_combined.predict(X_test)
    y_train_prob = rf_combined.predict_proba(X_train)[:, 1]
    y_test_prob  = rf_combined.predict_proba(X_test)[:, 1]

    # ── Metrics ───────────────────────────────────────────────────────────────
    metrics = {}
    for split, y_true, y_pred, y_prob in [
        ('train', y_train, y_train_pred, y_train_prob),
        ('test',  y_test,  y_test_pred,  y_test_prob)
    ]:
        metrics[f'{split}_accuracy']  = accuracy_score(y_true, y_pred)
        metrics[f'{split}_precision'] = precision_score(y_true, y_pred, zero_division=0)
        metrics[f'{split}_recall']    = recall_score(y_true, y_pred, zero_division=0)
        metrics[f'{split}_f1']        = f1_score(y_true, y_pred, zero_division=0)
        metrics[f'{split}_auc']       = roc_auc_score(y_true, y_prob)

    mlflow.log_metrics(metrics)

    # ── Print performance ─────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("RANDOM FOREST PERFORMANCE (FTS + SAL COMBINED)")
    print("=" * 70)
    for split, y_true, y_pred in [('TRAIN', y_train, y_train_pred),
                                   ('TEST',  y_test,  y_test_pred)]:
        print(f"\n{'─'*30} {split} {'─'*30}")
        print(f"  Accuracy:  {metrics[f'{split.lower()}_accuracy']:.4f}")
        print(f"  Precision: {metrics[f'{split.lower()}_precision']:.4f}")
        print(f"  Recall:    {metrics[f'{split.lower()}_recall']:.4f}")
        print(f"  F1 Score:  {metrics[f'{split.lower()}_f1']:.4f}")
        print(f"  AUC-ROC:   {metrics[f'{split.lower()}_auc']:.4f}")
        print(f"\n  Classification Report:")
        print(classification_report(y_true, y_pred, target_names=['Good (0)', 'Bad (1)']))

    # ── Overfitting check ─────────────────────────────────────────────────────
    auc_gap = metrics['train_auc'] - metrics['test_auc']
    f1_gap  = metrics['train_f1']  - metrics['test_f1']
    print(f"\n{'='*70}")
    print(f"OVERFITTING CHECK")
    print(f"{'='*70}")
    print(f"  Train AUC: {metrics['train_auc']:.4f}  |  Test AUC: {metrics['test_auc']:.4f}  |  Gap: {auc_gap:.4f}")
    print(f"  Train F1:  {metrics['train_f1']:.4f}  |  Test F1:  {metrics['test_f1']:.4f}  |  Gap: {f1_gap:.4f}")
    if auc_gap > 0.05:
        print(f"  ⚠️  AUC gap > 0.05 — potential overfitting")
    else:
        print(f"  ✅  AUC gap ≤ 0.05 — no significant overfitting")

    # ── Feature Importance ────────────────────────────────────────────────────
    feat_imp_combined = pd.DataFrame({
        'feature': all_features,
        'importance': rf_combined.feature_importances_
    }).sort_values('importance', ascending=False).reset_index(drop=True)

    # Tag each feature as FTS or SAL
    feat_imp_combined['source'] = feat_imp_combined['feature'].apply(
        lambda f: 'SAL' if f in sal_features else 'FTS'
    )

    print(f"\n{'='*70}")
    print(f"FEATURE IMPORTANCE (Gini / MDI)")
    print(f"{'='*70}")
    for _, row in feat_imp_combined.iterrows():
        bar = '█' * int(row['importance'] * 100)
        print(f"  [{row['source']:3s}] {row['feature']:35s} {row['importance']:.4f}  {bar}")

    # ── Aggregate importance by source ────────────────────────────────────────
    sal_total_imp = feat_imp_combined.loc[feat_imp_combined['source'] == 'SAL', 'importance'].sum()
    fts_total_imp = feat_imp_combined.loc[feat_imp_combined['source'] == 'FTS', 'importance'].sum()
    print(f"\n{'='*70}")
    print(f"AGGREGATE IMPORTANCE BY FEATURE GROUP")
    print(f"{'='*70}")
    print(f"  FTS features total importance: {fts_total_imp:.4f}  ({fts_total_imp*100:.1f}%)")
    print(f"  SAL features total importance: {sal_total_imp:.4f}  ({sal_total_imp*100:.1f}%)")

    # Top SAL features
    top_sal = feat_imp_combined[feat_imp_combined['source'] == 'SAL'].head(5)
    print(f"\n  Top 5 SAL features:")
    for _, row in top_sal.iterrows():
        print(f"    {row['feature']:35s} {row['importance']:.4f}")

    # ── Recommendation ────────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"RECOMMENDATION: KEEP SAL FEATURES?")
    print(f"{'='*70}")
    if sal_total_imp >= 0.10:
        print(f"  ✅ YES — SAL features contribute {sal_total_imp*100:.1f}% of total importance.")
        print(f"     They add meaningful signal beyond FTS flow features.")
    elif sal_total_imp >= 0.05:
        print(f"  🟡 MARGINAL — SAL features contribute {sal_total_imp*100:.1f}% of total importance.")
        print(f"     Consider keeping only the top SAL features and dropping low-importance ones.")
    else:
        print(f"  ❌ NO — SAL features contribute only {sal_total_imp*100:.1f}% of total importance.")
        print(f"     They add negligible signal. FTS features alone are sufficient.")

    # ── Plots ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(24, 8))

    # 1. Feature importance bar chart (color-coded by source)
    ax = axes[0]
    colors = ['#e07b39' if s == 'SAL' else 'steelblue' for s in feat_imp_combined['source'][::-1]]
    ax.barh(feat_imp_combined['feature'][::-1], feat_imp_combined['importance'][::-1], color=colors)
    ax.set_xlabel('Importance (Gini)')
    ax.set_title('Feature Importance (Blue=FTS, Orange=SAL)')

    # 2. Confusion matrix – Test
    ax = axes[1]
    cm = confusion_matrix(y_test, y_test_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Good', 'Bad'], yticklabels=['Good', 'Bad'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'Test Confusion Matrix (AUC={metrics["test_auc"]:.3f})')

    # 3. ROC Curve – Train & Test
    ax = axes[2]
    for label, y_true, y_prob, ls in [
        ('Train', y_train, y_train_prob, '--'),
        ('Test',  y_test,  y_test_prob,  '-')
    ]:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_val = roc_auc_score(y_true, y_prob)
        ax.plot(fpr, tpr, ls, label=f'{label} (AUC={auc_val:.3f})')
    ax.plot([0, 1], [0, 1], 'k:', alpha=0.4)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('ROC Curve'); ax.legend()

    plt.tight_layout()
    mlflow.log_figure(fig, 'rf_fts_sal_combined_performance.png')
    plt.show()

    print(f"\n✅ MLflow run logged: {run.info.run_id}")

display(feat_imp_combined)

In [ ]:
universe_sal.columns
df_thin.head()

In [ ]:
# ── Pull ALL columns from AECB feature store for Thin File refs ──────────────

# Get unique REFERENCE_NUMBERs from universe_sal
ref_nums = universe_sal['REFERENCE_NUMBER'].dropna().unique().tolist()
ref_nums = [str(int(float(r))) if r not in [None, '', 'nan'] else r for r in ref_nums]
print(f"Unique REFERENCE_NUMBERs in universe_sal: {len(ref_nums):,}")

# Pull full AECB feature store (all columns) filtered to Thin File + matching refs
aecb_full = spark.sql("""
    SELECT *
    FROM mis_automation.silver_bureauone.slv_credit_card_aecb_feature_store
""")

aecb_full_filtered = (
    aecb_full
    .withColumn("Reference_Number", F.col("Reference_Number").cast("string"))
    .filter(F.col("Reference_Number").isin(ref_nums))
    .filter(F.col("file_type") == "Thin File")
)

print(f"AECB rows matched: {aecb_full_filtered.count():,}")

# Convert to pandas
aecb_pd = aecb_full_filtered.toPandas()
aecb_pd['Reference_Number'] = aecb_pd['Reference_Number'].astype(str)

# ── Left join universe_sal with AECB on REFERENCE_NUMBER ─────────────────────
universe_sal['REFERENCE_NUMBER'] = universe_sal['REFERENCE_NUMBER'].astype(str)

universe_sal_aecb = universe_sal.merge(
    aecb_pd,
    left_on='REFERENCE_NUMBER',
    right_on='Reference_Number',
    how='left',
    suffixes=('', '_aecb')
)

# ── List all AECB columns ────────────────────────────────────────────────────
aecb_cols = [c for c in aecb_pd.columns if c != 'Reference_Number']
print(f"\nAECB columns ({len(aecb_cols)}):")
for i, col in enumerate(aecb_cols, 1):
    print(f"  {i:>3}. {col}")

print(f"\nuniverse_sal_aecb shape: {universe_sal_aecb.shape}")
print(f"AECB coverage: {universe_sal_aecb['file_type'].notna().sum():,} / {len(universe_sal_aecb):,} rows ({universe_sal_aecb['file_type'].notna().mean():.1%})")

In [ ]:
# ── Random Forest: AECB features → target ────────────────────────────────────
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, f1_score, precision_score, recall_score, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── Feature list ──────────────────────────────────────────────────────────────
aecb_features = [
    # Bureau score & rating
    'AECB_Score', 'score_missing_flag',
    # Facility counts
    'active_facility_count', 'closed_facility_count', 'Active_noof_prds',
    'active_card_count', 'active_personal_loan_count',
    # CC utilization
    'Max_CC_utilization', 'Min_CC_utilization', 'Mean_CC_utilization',
    'total_revolving_utilization',
    # DPD features
    'max_dpd_3m', 'max_dpd_6m', 'max_dpd_12m', 'max_dpd_24m', 'max_dpd_36m',
    'ever_30dpd_12m', 'ever_60dpd_24m', 'ever_90dpd_36m',
    'months_since_last_dpd', 'consecutive_dpd_months_max',
    # Inquiry features
    'inquiry_count_7d', 'inquiry_count_30d', 'inquiry_count_90d',
    'inquiry_count_180d', 'inquiry_count_365d',
    'inquiry_velocity_30d_vs_12m', 'days_since_last_inquiry',
    # Income & debt ratios
    'total_payment_to_income', 'total_balance_to_income',
    'external_debt_to_income', 'revolving_limit_to_income',
    # New facility activity
    'new_facility_count_3m', 'new_facility_count_6m', 'new_facility_count_12m',
    'new_cc_count_6m', 'new_cc_count_12m', 'new_cc_count_24m',
]

print(f"Total AECB features: {len(aecb_features)}")

# ── Check which features exist in the dataframe ──────────────────────────────
missing_feats = [f for f in aecb_features if f not in universe_sal_aecb.columns]
if missing_feats:
    print(f"⚠️  Missing features (will be skipped): {missing_feats}")
    aecb_features = [f for f in aecb_features if f in universe_sal_aecb.columns]
    print(f"Using {len(aecb_features)} available features")

# ── Filter to rows with valid target ──────────────────────────────────────────
df_model_aecb = universe_sal_aecb[universe_sal_aecb['target'].isin([0, 1])].copy()

# ── Deduplicate to ONE row per card (latest snapshot) ─────────────────────────
df_model_aecb = df_model_aecb.sort_values('SNAPSHOT_DATE', ascending=False)
df_model_aecb = df_model_aecb.drop_duplicates(subset='CREDITCARDNO', keep='first').reset_index(drop=True)

X = df_model_aecb[aecb_features].fillna(-1)
y = df_model_aecb['target']
print(f"After dedup to 1 row/card: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Target distribution:\n{y.value_counts().sort_index()}")
print(f"Bad rate: {y.mean():.4f}")

# ── Card-level Train / Test split (same setup as cell 14) ─────────────────────
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df_model_aecb['CREDITCARDNO']))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print(f"\nTrain: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
print(f"Train bad rate: {y_train.mean():.4f}  |  Test bad rate: {y_test.mean():.4f}")
print(f"Card overlap: {len(set(df_model_aecb.loc[train_idx, 'CREDITCARDNO']) & set(df_model_aecb.loc[test_idx, 'CREDITCARDNO']))} (should be 0)")

# ── Train Random Forest ───────────────────────────────────────────────────────
mlflow.sklearn.autolog(log_models=True, silent=True)

with mlflow.start_run(run_name="RF_AECB_features") as run:
    rf_aecb = RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    rf_aecb.fit(X_train, y_train)

    # ── Predictions ───────────────────────────────────────────────────────────
    y_train_pred = rf_aecb.predict(X_train)
    y_test_pred  = rf_aecb.predict(X_test)
    y_train_prob = rf_aecb.predict_proba(X_train)[:, 1]
    y_test_prob  = rf_aecb.predict_proba(X_test)[:, 1]

    # ── Metrics ───────────────────────────────────────────────────────────────
    metrics = {}
    for split, y_true, y_pred, y_prob in [
        ('train', y_train, y_train_pred, y_train_prob),
        ('test',  y_test,  y_test_pred,  y_test_prob)
    ]:
        metrics[f'{split}_accuracy']  = accuracy_score(y_true, y_pred)
        metrics[f'{split}_precision'] = precision_score(y_true, y_pred, zero_division=0)
        metrics[f'{split}_recall']    = recall_score(y_true, y_pred, zero_division=0)
        metrics[f'{split}_f1']        = f1_score(y_true, y_pred, zero_division=0)
        metrics[f'{split}_auc']       = roc_auc_score(y_true, y_prob)

    mlflow.log_metrics(metrics)

    # ── Print performance ─────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("RANDOM FOREST PERFORMANCE (AECB FEATURES)")
    print("=" * 70)
    for split, y_true, y_pred in [('TRAIN', y_train, y_train_pred),
                                   ('TEST',  y_test,  y_test_pred)]:
        print(f"\n{'─'*30} {split} {'─'*30}")
        print(f"  Accuracy:  {metrics[f'{split.lower()}_accuracy']:.4f}")
        print(f"  Precision: {metrics[f'{split.lower()}_precision']:.4f}")
        print(f"  Recall:    {metrics[f'{split.lower()}_recall']:.4f}")
        print(f"  F1 Score:  {metrics[f'{split.lower()}_f1']:.4f}")
        print(f"  AUC-ROC:   {metrics[f'{split.lower()}_auc']:.4f}")
        print(f"\n  Classification Report:")
        print(classification_report(y_true, y_pred, target_names=['Good (0)', 'Bad (1)']))

    # ── Overfitting check ─────────────────────────────────────────────────────
    auc_gap = metrics['train_auc'] - metrics['test_auc']
    f1_gap  = metrics['train_f1']  - metrics['test_f1']
    print(f"\n{'='*70}")
    print(f"OVERFITTING CHECK")
    print(f"{'='*70}")
    print(f"  Train AUC: {metrics['train_auc']:.4f}  |  Test AUC: {metrics['test_auc']:.4f}  |  Gap: {auc_gap:.4f}")
    print(f"  Train F1:  {metrics['train_f1']:.4f}  |  Test F1:  {metrics['test_f1']:.4f}  |  Gap: {f1_gap:.4f}")
    if auc_gap > 0.05:
        print(f"  ⚠️  AUC gap > 0.05 — potential overfitting")
    else:
        print(f"  ✅  AUC gap ≤ 0.05 — no significant overfitting")

    # ── Feature Importance ────────────────────────────────────────────────────
    feat_imp_aecb = pd.DataFrame({
        'feature': aecb_features,
        'importance': rf_aecb.feature_importances_
    }).sort_values('importance', ascending=False).reset_index(drop=True)

    print(f"\n{'='*70}")
    print(f"FEATURE IMPORTANCE (Gini / MDI)")
    print(f"{'='*70}")
    for _, row in feat_imp_aecb.iterrows():
        bar = '█' * int(row['importance'] * 100)
        print(f"  {row['feature']:40s} {row['importance']:.4f}  {bar}")

    # ── Plots ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(26, 9))

    # 1. Feature importance bar chart
    ax = axes[0]
    top_n = feat_imp_aecb.head(20)  # show top 20 for readability
    ax.barh(top_n['feature'][::-1], top_n['importance'][::-1], color='teal')
    ax.set_xlabel('Importance (Gini)')
    ax.set_title('Top 20 AECB Feature Importance')

    # 2. Confusion matrix – Test
    ax = axes[1]
    cm = confusion_matrix(y_test, y_test_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Good', 'Bad'], yticklabels=['Good', 'Bad'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'Test Confusion Matrix (AUC={metrics["test_auc"]:.3f})')

    # 3. ROC Curve – Train & Test
    ax = axes[2]
    for label, y_true, y_prob, ls in [
        ('Train', y_train, y_train_prob, '--'),
        ('Test',  y_test,  y_test_prob,  '-')
    ]:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_val = roc_auc_score(y_true, y_prob)
        ax.plot(fpr, tpr, ls, label=f'{label} (AUC={auc_val:.3f})')
    ax.plot([0, 1], [0, 1], 'k:', alpha=0.4)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('ROC Curve'); ax.legend()

    plt.tight_layout()
    mlflow.log_figure(fig, 'rf_aecb_performance.png')
    plt.show()

    print(f"\n✅ MLflow run logged: {run.info.run_id}")

display(feat_imp_aecb)

In [ ]:
# ── Random Forest: AECB + FTS + SAL Combined → target ────────────────────────
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, f1_score, precision_score, recall_score, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── Feature lists ─────────────────────────────────────────────────────────────
fts_features = [
    'total_inflow', 'total_outflow', 'total_net_flow',
    'avg_monthly_inflow', 'avg_monthly_outflow', 'avg_monthly_net_flow',
    'std_monthly_inflow', 'std_monthly_outflow', 'std_monthly_net_flow',
    'max_monthly_inflow', 'max_monthly_outflow',
    'min_monthly_inflow', 'min_monthly_outflow',
    'total_inflow_txn_count', 'total_outflow_txn_count',
    'avg_monthly_inflow_txns', 'avg_monthly_outflow_txns',
    'n_active_months', 'outflow_inflow_ratio'
]

sal_features = [
    'num_sal_months', 'mean_sal_amount', 'std_sal_amount', 'var_sal_amount',
    'min_sal_amount', 'max_sal_amount', 'mean_posting_day', 'std_posting_day',
    'var_posting_day', 'min_posting_day', 'max_posting_day',
    'cv_sal_amount', 'cv_posting_day', 'posting_day_range'
]

aecb_features = [
    'AECB_Score', 'score_missing_flag',
    'active_facility_count', 'closed_facility_count', 'Active_noof_prds',
    'active_card_count', 'active_personal_loan_count',
    'Max_CC_utilization', 'Min_CC_utilization', 'Mean_CC_utilization',
    'total_revolving_utilization',
    'max_dpd_3m', 'max_dpd_6m', 'max_dpd_12m', 'max_dpd_24m', 'max_dpd_36m',
    'ever_30dpd_12m', 'ever_60dpd_24m', 'ever_90dpd_36m',
    'months_since_last_dpd', 'consecutive_dpd_months_max',
    'inquiry_count_7d', 'inquiry_count_30d', 'inquiry_count_90d',
    'inquiry_count_180d', 'inquiry_count_365d',
    'inquiry_velocity_30d_vs_12m', 'days_since_last_inquiry',
    'total_payment_to_income', 'total_balance_to_income',
    'external_debt_to_income', 'revolving_limit_to_income',
    'new_facility_count_3m', 'new_facility_count_6m', 'new_facility_count_12m',
    'new_cc_count_6m', 'new_cc_count_12m', 'new_cc_count_24m',
]

# ── Check which features exist ────────────────────────────────────────────────
all_combined_features = fts_features + sal_features + aecb_features
missing_feats = [f for f in all_combined_features if f not in universe_sal_aecb.columns]
if missing_feats:
    print(f"⚠️  Missing features (will be skipped): {missing_feats}")
    all_combined_features = [f for f in all_combined_features if f in universe_sal_aecb.columns]

# Tag each feature by source
feature_source = {}
for f in all_combined_features:
    if f in fts_features:
        feature_source[f] = 'FTS'
    elif f in sal_features:
        feature_source[f] = 'SAL'
    else:
        feature_source[f] = 'AECB'

print(f"Total combined features: {len(all_combined_features)}")
print(f"  FTS:  {sum(1 for v in feature_source.values() if v == 'FTS')}")
print(f"  SAL:  {sum(1 for v in feature_source.values() if v == 'SAL')}")
print(f"  AECB: {sum(1 for v in feature_source.values() if v == 'AECB')}")

# ── Filter to rows with valid target ──────────────────────────────────────────
df_model_full = universe_sal_aecb[universe_sal_aecb['target'].isin([0, 1])].copy()

# ── Deduplicate to ONE row per card (latest snapshot) ─────────────────────────
df_model_full = df_model_full.sort_values('SNAPSHOT_DATE', ascending=False)
df_model_full = df_model_full.drop_duplicates(subset='CREDITCARDNO', keep='first').reset_index(drop=True)

X = df_model_full[all_combined_features].fillna(-1)
y = df_model_full['target']
print(f"\nAfter dedup to 1 row/card: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Target distribution:\n{y.value_counts().sort_index()}")
print(f"Bad rate: {y.mean():.4f}")

# ── Card-level Train / Test split ─────────────────────────────────────────────
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df_model_full['CREDITCARDNO']))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print(f"\nTrain: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
print(f"Train bad rate: {y_train.mean():.4f}  |  Test bad rate: {y_test.mean():.4f}")
print(f"Card overlap: {len(set(df_model_full.loc[train_idx, 'CREDITCARDNO']) & set(df_model_full.loc[test_idx, 'CREDITCARDNO']))} (should be 0)")

# ── Train Random Forest ───────────────────────────────────────────────────────
mlflow.sklearn.autolog(log_models=True, silent=True)

with mlflow.start_run(run_name="RF_AECB_FTS_SAL_combined") as run:
    rf_full = RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt',
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    rf_full.fit(X_train, y_train)

    # ── Predictions ───────────────────────────────────────────────────────────
    y_train_pred = rf_full.predict(X_train)
    y_test_pred  = rf_full.predict(X_test)
    y_train_prob = rf_full.predict_proba(X_train)[:, 1]
    y_test_prob  = rf_full.predict_proba(X_test)[:, 1]

    # ── Metrics ───────────────────────────────────────────────────────────────
    metrics = {}
    for split, y_true, y_pred, y_prob in [
        ('train', y_train, y_train_pred, y_train_prob),
        ('test',  y_test,  y_test_pred,  y_test_prob)
    ]:
        metrics[f'{split}_accuracy']  = accuracy_score(y_true, y_pred)
        metrics[f'{split}_precision'] = precision_score(y_true, y_pred, zero_division=0)
        metrics[f'{split}_recall']    = recall_score(y_true, y_pred, zero_division=0)
        metrics[f'{split}_f1']        = f1_score(y_true, y_pred, zero_division=0)
        metrics[f'{split}_auc']       = roc_auc_score(y_true, y_prob)

    mlflow.log_metrics(metrics)

    # ── Print performance ─────────────────────────────────────────────────────
    print("\n" + "=" * 70)
    print("RANDOM FOREST PERFORMANCE (AECB + FTS + SAL COMBINED)")
    print("=" * 70)
    for split, y_true, y_pred in [('TRAIN', y_train, y_train_pred),
                                   ('TEST',  y_test,  y_test_pred)]:
        print(f"\n{'─'*30} {split} {'─'*30}")
        print(f"  Accuracy:  {metrics[f'{split.lower()}_accuracy']:.4f}")
        print(f"  Precision: {metrics[f'{split.lower()}_precision']:.4f}")
        print(f"  Recall:    {metrics[f'{split.lower()}_recall']:.4f}")
        print(f"  F1 Score:  {metrics[f'{split.lower()}_f1']:.4f}")
        print(f"  AUC-ROC:   {metrics[f'{split.lower()}_auc']:.4f}")
        print(f"\n  Classification Report:")
        print(classification_report(y_true, y_pred, target_names=['Good (0)', 'Bad (1)']))

    # ── Overfitting check ─────────────────────────────────────────────────────
    auc_gap = metrics['train_auc'] - metrics['test_auc']
    f1_gap  = metrics['train_f1']  - metrics['test_f1']
    print(f"\n{'='*70}")
    print(f"OVERFITTING CHECK")
    print(f"{'='*70}")
    print(f"  Train AUC: {metrics['train_auc']:.4f}  |  Test AUC: {metrics['test_auc']:.4f}  |  Gap: {auc_gap:.4f}")
    print(f"  Train F1:  {metrics['train_f1']:.4f}  |  Test F1:  {metrics['test_f1']:.4f}  |  Gap: {f1_gap:.4f}")
    if auc_gap > 0.05:
        print(f"  ⚠️  AUC gap > 0.05 — potential overfitting")
    else:
        print(f"  ✅  AUC gap ≤ 0.05 — no significant overfitting")

    # ── Feature Importance ────────────────────────────────────────────────────
    feat_imp_full = pd.DataFrame({
        'feature': all_combined_features,
        'importance': rf_full.feature_importances_
    }).sort_values('importance', ascending=False).reset_index(drop=True)

    feat_imp_full['source'] = feat_imp_full['feature'].map(feature_source)

    print(f"\n{'='*70}")
    print(f"FEATURE IMPORTANCE (Gini / MDI)")
    print(f"{'='*70}")
    for _, row in feat_imp_full.iterrows():
        bar = '█' * int(row['importance'] * 100)
        print(f"  [{row['source']:4s}] {row['feature']:40s} {row['importance']:.4f}  {bar}")

    # ── Aggregate importance by source ────────────────────────────────────────
    imp_by_source = feat_imp_full.groupby('source')['importance'].sum().sort_values(ascending=False)
    print(f"\n{'='*70}")
    print(f"AGGREGATE IMPORTANCE BY FEATURE GROUP")
    print(f"{'='*70}")
    for src, imp in imp_by_source.items():
        print(f"  {src:4s} features total importance: {imp:.4f}  ({imp*100:.1f}%)")

    # ── Plots ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(28, 10))

    # 1. Feature importance bar chart (color-coded by source)
    ax = axes[0]
    top_n = feat_imp_full.head(25)
    color_map = {'FTS': 'steelblue', 'SAL': '#e07b39', 'AECB': 'teal'}
    colors = [color_map.get(s, 'gray') for s in top_n['source'][::-1]]
    ax.barh(top_n['feature'][::-1], top_n['importance'][::-1], color=colors)
    ax.set_xlabel('Importance (Gini)')
    ax.set_title('Top 25 Feature Importance (Blue=FTS, Orange=SAL, Teal=AECB)')

    # 2. Confusion matrix – Test
    ax = axes[1]
    cm = confusion_matrix(y_test, y_test_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Good', 'Bad'], yticklabels=['Good', 'Bad'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'Test Confusion Matrix (AUC={metrics["test_auc"]:.3f})')

    # 3. ROC Curve – Train & Test
    ax = axes[2]
    for label, y_true, y_prob, ls in [
        ('Train', y_train, y_train_prob, '--'),
        ('Test',  y_test,  y_test_prob,  '-')
    ]:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_val = roc_auc_score(y_true, y_prob)
        ax.plot(fpr, tpr, ls, label=f'{label} (AUC={auc_val:.3f})')
    ax.plot([0, 1], [0, 1], 'k:', alpha=0.4)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('ROC Curve'); ax.legend()

    plt.tight_layout()
    mlflow.log_figure(fig, 'rf_aecb_fts_sal_combined_performance.png')
    plt.show()

    print(f"\n✅ MLflow run logged: {run.info.run_id}")

display(feat_imp_full)

In [ ]:
# ── Good Push: Clean commit to cc_eda ─────────────────────────────────────────
import base64, json, os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat, ImportFormat

w = WorkspaceClient()

# Step 1: Export this notebook as Jupyter
resp = w.workspace.export(
    path="/Users/zheng.zhu@rakbank.ae/FTS_Data_Analysis",
    format=ExportFormat.JUPYTER
)
nb_bytes = base64.b64decode(resp.content)
with open("/tmp/notebook_export.ipynb", "wb") as f:
    f.write(nb_bytes)
print(f"Exported: {os.path.getsize('/tmp/notebook_export.ipynb'):,} bytes")

# Step 2: Strip execution outputs (prevents GIT_SENSITIVE_TOKEN_DETECTED)
with open("/tmp/notebook_export.ipynb", "r") as f:
    nb = json.load(f)
for cell in nb['cells']:
    cell['outputs'] = []
    cell['execution_count'] = None
with open("/tmp/notebook_clean.ipynb", "w") as f:
    json.dump(nb, f, indent=1)
orig = os.path.getsize("/tmp/notebook_export.ipynb")
clean = os.path.getsize("/tmp/notebook_clean.ipynb")
print(f"Original: {orig:,} bytes | Clean: {clean:,} bytes ({100 - clean/orig*100:.1f}% smaller)")

# Step 3: Import clean notebook into Git folder
with open("/tmp/notebook_clean.ipynb", "rb") as f:
    content = base64.b64encode(f.read()).decode("utf-8")
w.workspace.import_(
    path="/Users/zheng.zhu@rakbank.ae/credit_card_application_playground/FTS_Data_Analysis",
    content=content,
    format=ImportFormat.JUPYTER,
    overwrite=True
)
print("Clean notebook imported into Git folder.")